# 🏠 Airbnb NYC Data Cleaning Pipeline

**Dataset:** 102,599 Airbnb listings in New York City  
**Goal:** Clean raw, messy data and export a polished CSV ready for **Tableau analysis**.

### Cleaning Steps
1. Drop `license` column (99.99% null — useless)
2. Drop `house_rules` (50%+ null, not needed for analysis)
3. Strip `$` and commas from `price` & `service fee` → convert to float
4. Fix typos in `neighbourhood group`: `"brookln"` → `"Brooklyn"`, `"manhatan"` → `"Manhattan"`
5. Parse `last review` as datetime
6. Drop rows where `price` is null (only 247 rows — safe to drop)
7. Fill moderate nulls in numeric columns (median / 0)
8. Fill remaining text/categorical nulls with sensible defaults
9. Drop rows with unfillable nulls (lat/long/neighbourhood — tiny count)
10. Standardize column names → lowercase + underscores
11. Tableau prep: remove bogus sentinel dates, fix `construction_year` dtype
12. Export → `cleaned_airbnb_final.csv`

---
## 1 · Import Libraries & Load Data

In [40]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('Airbnb_Open_Data.csv', low_memory=False)
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

Loaded 102,599 rows × 26 columns


,id,NAME,host id,host_identity_verified,host name,neighbourhood group,neighbourhood,lat,long,country,...,service fee,minimum nights,number of reviews,last review,reviews per month,review rate number,calculated host listings count,availability 365,house_rules,license
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,$193,10.0,9.0,10/19/2021,0.21,4.0,6.0,286.0,Clean up and treat the home the way you'd like...,NaN
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,$28,30.0,45.0,5/21/2022,0.38,4.0,2.0,228.0,Pet friendly but please confirm with me if the...,NaN
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,NaN,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,$124,3.0,0.0,NaN,NaN,5.0,1.0,352.0,"I encourage you to use my kitchen, cooking and...",NaN


---
## 2 · Initial Inspection

In [41]:
# Data types & non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102599 entries, 0 to 102598
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   id                              102599 non-null  int64  
 1   NAME                            102349 non-null  object 
 2   host id                         102599 non-null  int64  
 3   host_identity_verified          102310 non-null  object 
 4   host name                       102193 non-null  object 
 5   neighbourhood group             102570 non-null  object 
 6   neighbourhood                   102583 non-null  object 
 7   lat                             102591 non-null  float64
 8   long                            102591 non-null  float64
 9   country                         102067 non-null  object 
 10  country code                    102468 non-null  object 
 11  instant_bookable                102494 non-null  object 
 12  cancellation_pol

In [42]:
# Null percentage per column
null_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
print("Columns with nulls (% missing):")
print(null_pct.to_string())

Columns with nulls (% missing):
license                           100.00
house_rules                        50.81
last review                        15.49
reviews per month                  15.48
country                             0.52
availability 365                    0.44
host name                           0.40
minimum nights                      0.40
review rate number                  0.32
calculated host listings count      0.31
host_identity_verified              0.28
service fee                         0.27
NAME                                0.24
price                               0.24
Construction year                   0.21
number of reviews                   0.18
country code                        0.13
instant_bookable                    0.10
cancellation_policy                 0.07
neighbourhood group                 0.03
neighbourhood                       0.02
long                                0.01
lat                                 0.01


---
## 3 · Drop Useless Columns

- **`license`** — 99.99% null, adds zero analytical value  
- **`house_rules`** — 50%+ null, free-text, not needed for quantitative analysis

In [43]:
print(f"Before: {df.shape[1]} columns")

df = df.drop(columns=['license', 'house_rules'])

print(f"After:  {df.shape[1]} columns  (dropped 'license' & 'house_rules')")

Before: 26 columns
After:  24 columns  (dropped 'license' & 'house_rules')


---
## 4 · Clean Currency Fields (`price` & `service fee`)

Both columns are stored as strings with `$` signs and commas → strip them and convert to `float`.

In [44]:
# Preview before cleaning
print("Before:")
print(df[['price', 'service fee']].head())
print(f"\nprice dtype: {df['price'].dtype}")
print(f"service fee dtype: {df['service fee'].dtype}")

Before:
   price service fee
0  $966        $193 
1  $142         $28 
2  $620        $124 
3  $368         $74 
4  $204         $41 

price dtype: object
service fee dtype: object


In [45]:
def clean_currency(series):
    """Strip $ and commas, convert to float."""
    return (
        series
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .str.strip()
        .replace('nan', np.nan)
        .astype(float)
    )

df['price'] = clean_currency(df['price'])
df['service fee'] = clean_currency(df['service fee'])

print("After:")
print(df[['price', 'service fee']].head())
print(f"\nprice dtype: {df['price'].dtype}")
print(f"service fee dtype: {df['service fee'].dtype}")

After:
   price  service fee
0  966.0        193.0
1  142.0         28.0
2  620.0        124.0
3  368.0         74.0
4  204.0         41.0

price dtype: float64
service fee dtype: float64


---
## 5 · Fix Typos in `neighbourhood group`

| Typo | Correction |
|---|---|
| `brookln` | `Brooklyn` |
| `manhatan` | `Manhattan` |

In [46]:
print("Before:")
print(df['neighbourhood group'].value_counts())

typo_map = {
    'brookln': 'Brooklyn',
    'manhatan': 'Manhattan'
}
df['neighbourhood group'] = df['neighbourhood group'].replace(typo_map)

print("\nAfter:")
print(df['neighbourhood group'].value_counts())

Before:
neighbourhood group
Manhattan        43792
Brooklyn         41842
Queens           13267
Bronx             2712
Staten Island      955
brookln              1
manhatan             1
Name: count, dtype: int64

After:
neighbourhood group
Manhattan        43793
Brooklyn         41843
Queens           13267
Bronx             2712
Staten Island      955
Name: count, dtype: int64


---
## 6 · Parse `last review` as Datetime

In [47]:
print(f"Before — dtype: {df['last review'].dtype}")

df['last review'] = pd.to_datetime(df['last review'], format='mixed', errors='coerce')

print(f"After  — dtype: {df['last review'].dtype}")
print(df['last review'].head())

Before — dtype: object
After  — dtype: datetime64[ns]
0   2021-10-19
1   2022-05-21
2          NaT
3   2019-07-05
4   2018-11-19
Name: last review, dtype: datetime64[ns]


---
## 7 · Drop Rows Where `price` is Null

Only **247 rows** (~0.24%) — safe to drop without data loss concerns.

In [48]:
null_price_count = df['price'].isnull().sum()
print(f"Rows with null price: {null_price_count}")

df = df.dropna(subset=['price'])

print(f"Rows remaining: {len(df):,}")

Rows with null price: 247
Rows remaining: 102,352


---
## 8 · Handle Numeric Nulls (Median / Zero Fill)

| Column | Strategy |
|---|---|
| `minimum nights` | Fill with **median** (right-skewed distribution) |
| `availability 365` | Fill with **median** |
| `review rate number` | Fill with **median** |
| `calculated host listings count` | Fill with **median** |
| `service fee` | Fill with **median** |
| `Construction year` | Fill with **median** |
| `number of reviews` | Fill with **0** (no reviews = 0) |
| `reviews per month` | Fill with **0** |

In [49]:
# Columns to fill with median
median_fill_cols = [
    'minimum nights',
    'availability 365',
    'review rate number',
    'calculated host listings count',
    'service fee',
    'Construction year'
]

for col in median_fill_cols:
    median_val = df[col].median()
    null_count = df[col].isnull().sum()
    df[col] = df[col].fillna(median_val)
    print(f"{col}: filled {null_count} nulls with median = {median_val}")

# Columns to fill with 0
zero_fill_cols = ['number of reviews', 'reviews per month']

for col in zero_fill_cols:
    null_count = df[col].isnull().sum()
    df[col] = df[col].fillna(0)
    print(f"{col}: filled {null_count} nulls with 0")

minimum nights: filled 409 nulls with median = 3.0
availability 365: filled 448 nulls with median = 96.0
review rate number: filled 326 nulls with median = 3.0
calculated host listings count: filled 319 nulls with median = 1.0
service fee: filled 239 nulls with median = 125.0
Construction year: filled 210 nulls with median = 2012.0
number of reviews: filled 183 nulls with 0
reviews per month: filled 15852 nulls with 0


---
## 9 · Handle Remaining Text & Categorical Nulls

| Column | Nulls | Strategy |
|---|---|---|
| `NAME` | 247 | Fill with **"Unnamed Listing"** |
| `host name` | 401 | Fill with **"Unknown"** |
| `host_identity_verified` | 285 | Fill with **"unconfirmed"** (conservative default) |
| `country` | 527 | Fill with **"United States"** (all listings are NYC) |
| `country code` | 126 | Fill with **"US"** |
| `instant_bookable` | 100 | Fill with **mode** → `False` |
| `cancellation_policy` | 71 | Fill with **mode** → `"moderate"` |

In [50]:
# --- Text columns: fill with meaningful defaults ---
text_fills = {
    'NAME': 'Unnamed Listing',
    'host name': 'Unknown',
    'host_identity_verified': 'unconfirmed',
    'country': 'United States',
    'country code': 'US'
}

for col, fill_val in text_fills.items():
    null_count = df[col].isnull().sum()
    df[col] = df[col].fillna(fill_val)
    print(f"{col}: filled {null_count} nulls with '{fill_val}'")

# --- Categorical columns: fill with mode ---
cat_fills = {
    'instant_bookable': df['instant_bookable'].mode()[0],
    'cancellation_policy': df['cancellation_policy'].mode()[0]
}

for col, fill_val in cat_fills.items():
    null_count = df[col].isnull().sum()
    df[col] = df[col].fillna(fill_val)
    print(f"{col}: filled {null_count} nulls with mode = '{fill_val}'")

NAME: filled 247 nulls with 'Unnamed Listing'
host name: filled 401 nulls with 'Unknown'
host_identity_verified: filled 285 nulls with 'unconfirmed'
country: filled 527 nulls with 'United States'
country code: filled 126 nulls with 'US'
instant_bookable: filled 100 nulls with mode = 'False'
cancellation_policy: filled 71 nulls with mode = 'moderate'


/var/folders/wf/6vvs9sv51zsfjmphw4rz66vw0000gn/T/ipykernel_8065/3780545073.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(fill_val)


---
## 10 · Drop Rows With Unfillable Nulls

These columns have very few nulls and can't be filled meaningfully:
- `neighbourhood group` — 28 rows (can't guess borough)
- `neighbourhood` — 15 rows
- `lat` / `long` — 8 rows (can't fabricate coordinates)

**Total dropped: ~50 rows (~0.05%)** — negligible data loss.

In [51]:
before = len(df)

drop_cols = ['neighbourhood group', 'neighbourhood', 'lat', 'long']
for col in drop_cols:
    null_count = df[col].isnull().sum()
    print(f"{col}: {null_count} null rows to drop")

df = df.dropna(subset=drop_cols)

after = len(df)
print(f"\nDropped {before - after} rows → {after:,} remaining")

neighbourhood group: 28 null rows to drop
neighbourhood: 15 null rows to drop
lat: 8 null rows to drop
long: 8 null rows to drop

Dropped 51 rows → 102,301 remaining


---
## 11 · Standardize Column Names

Convert all column names to **lowercase** with **underscores** replacing spaces.

In [52]:
print("Before:")
print(df.columns.tolist())

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

print("\nAfter:")
print(df.columns.tolist())

Before:
['id', 'NAME', 'host id', 'host_identity_verified', 'host name', 'neighbourhood group', 'neighbourhood', 'lat', 'long', 'country', 'country code', 'instant_bookable', 'cancellation_policy', 'room type', 'Construction year', 'price', 'service fee', 'minimum nights', 'number of reviews', 'last review', 'reviews per month', 'review rate number', 'calculated host listings count', 'availability 365']

After:
['id', 'name', 'host_id', 'host_identity_verified', 'host_name', 'neighbourhood_group', 'neighbourhood', 'lat', 'long', 'country', 'country_code', 'instant_bookable', 'cancellation_policy', 'room_type', 'construction_year', 'price', 'service_fee', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'review_rate_number', 'calculated_host_listings_count', 'availability_365']


---
## 12 · Tableau Prep Fixes

Two minor fixes before exporting for Tableau:

1. **`last_review`** — Replace bogus `1900-01-01` sentinel dates with `NaT` (Tableau handles NaT/blank dates gracefully)
2. **`construction_year`** — Convert from `float64` → `int64` (Tableau shows 2012.0 otherwise)

In [53]:
# Fix 1: Remove bogus sentinel dates (listings that were never reviewed)
bogus_count = (df['last_review'].dt.year < 2000).sum()
df.loc[df['last_review'].dt.year < 2000, 'last_review'] = pd.NaT
print(f"last_review: replaced {bogus_count} bogus 1900-01-01 dates with NaT")

# Fix 2: construction_year float → int
print(f"\nconstruction_year dtype before: {df['construction_year'].dtype}")
df['construction_year'] = df['construction_year'].astype(int)
print(f"construction_year dtype after:  {df['construction_year'].dtype}")

print(f"\nSample construction_year values: {df['construction_year'].head().tolist()}")

last_review: replaced 0 bogus 1900-01-01 dates with NaT

construction_year dtype before: float64
construction_year dtype after:  int64

Sample construction_year values: [2020, 2007, 2005, 2005, 2009]


---
## 13 · Final Inspection

In [54]:
print(f"Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\n--- Data Types ---")
print(df.dtypes)
print("\n--- Null Check ---")
nulls = df.isnull().sum()
nulls = nulls[nulls > 0]
if len(nulls) == 0:
    print("✅ ZERO NULLS across all columns!")
else:
    print(f"Nulls only in 'last_review' (never-reviewed listings — Tableau handles this):")
    print(nulls)

Final shape: 102,301 rows × 24 columns

--- Data Types ---
id                                         int64
name                                      object
host_id                                    int64
host_identity_verified                    object
host_name                                 object
neighbourhood_group                       object
neighbourhood                             object
lat                                      float64
long                                     float64
country                                   object
country_code                              object
instant_bookable                            bool
cancellation_policy                       object
room_type                                 object
construction_year                          int64
price                                    float64
service_fee                              float64
minimum_nights                           float64
number_of_reviews                        float64
last_revie

In [55]:
df.describe()

,id,host_id,lat,long,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
count,1.023010e+05,1.023010e+05,102301.000000,102301.000000,102301.000000,102301.000000,102301.000000,102301.000000,102301.00000,86439,102301.000000,102301.000000,102301.000000,102301.000000
mean,2.916731e+07,4.924450e+10,40.728086,-73.949646,2012.485724,625.276869,125.034936,8.102521,27.41417,2019-06-12 03:45:46.885086464,1.162280,3.278482,7.907841,140.913364
min,1.001254e+06,1.236005e+08,40.499790,-74.249840,2003.000000,50.000000,10.000000,-1223.000000,0.00000,2012-07-11 00:00:00,0.000000,1.000000,1.000000,-10.000000
25%,1.512420e+07,2.456871e+10,40.688720,-73.982570,2008.000000,340.000000,68.000000,2.000000,1.00000,2018-10-28 00:00:00,0.090000,2.000000,1.000000,3.000000
50%,2.915814e+07,4.910321e+10,40.722280,-73.954440,2012.000000,624.000000,125.000000,3.000000,7.00000,2019-06-14 00:00:00,0.480000,3.000000,1.000000,96.000000
75%,4.322246e+07,7.397591e+10,40.762760,-73.932350,2017.000000,913.000000,182.000000,5.000000,30.00000,2019-07-05 00:00:00,1.710000,4.000000,2.000000,268.000000
max,5.736742e+07,9.876313e+10,40.916970,-73.705220,2022.000000,1200.000000,240.000000,5645.000000,1024.00000,2058-06-16 00:00:00,90.000000,5.000000,332.000000,3677.000000
std,1.624843e+07,2.853720e+10,0.055872,0.049525,5.759321,331.683894,66.258401,30.496212,49.45125,NaN,1.682187,1.282476,32.170844,135.171237


In [56]:
df.head()

,id,name,host_id,host_identity_verified,host_name,neighbourhood_group,neighbourhood,lat,long,country,...,construction_year,price,service_fee,minimum_nights,number_of_reviews,last_review,reviews_per_month,review_rate_number,calculated_host_listings_count,availability_365
0,1001254,Clean & quiet apt home by the park,80014485718,unconfirmed,Madaline,Brooklyn,Kensington,40.64749,-73.97237,United States,...,2020,966.0,193.0,10.0,9.0,2021-10-19,0.21,4.0,6.0,286.0
1,1002102,Skylit Midtown Castle,52335172823,verified,Jenna,Manhattan,Midtown,40.75362,-73.98377,United States,...,2007,142.0,28.0,30.0,45.0,2022-05-21,0.38,4.0,2.0,228.0
2,1002403,THE VILLAGE OF HARLEM....NEW YORK !,78829239556,unconfirmed,Elise,Manhattan,Harlem,40.80902,-73.94190,United States,...,2005,620.0,124.0,3.0,0.0,NaT,0.00,5.0,1.0,352.0
3,1002755,Unnamed Listing,85098326012,unconfirmed,Garry,Brooklyn,Clinton Hill,40.68514,-73.95976,United States,...,2005,368.0,74.0,30.0,270.0,2019-07-05,4.64,4.0,1.0,322.0
4,1003689,Entire Apt: Spacious Studio/Loft by central park,92037596077,verified,Lyndon,Manhattan,East Harlem,40.79851,-73.94399,United States,...,2009,204.0,41.0,10.0,9.0,2018-11-19,0.10,3.0,1.0,289.0


---
## 14 · Export Final Cleaned Data

In [57]:
df.to_csv('cleaned_airbnb_final.csv', index=False)
print("✅ Exported → cleaned_airbnb_final.csv")
print(f"   Rows: {len(df):,}")
print(f"   Columns: {df.shape[1]}")
print(f"   Ready for Tableau! 🎯")

✅ Exported → cleaned_airbnb_final.csv
   Rows: 102,301
   Columns: 24
   Ready for Tableau! 🎯


---
## Summary of Changes

| Step | Action | Details |
|:---:|---|---|
| 1 | Dropped `license` column | 99.99% null |
| 2 | Dropped `house_rules` column | 50%+ null, not needed |
| 3 | Cleaned `price` & `service fee` | Stripped `$` and commas → float |
| 4 | Fixed neighbourhood group typos | `brookln` → `Brooklyn`, `manhatan` → `Manhattan` |
| 5 | Parsed `last review` | Converted to datetime |
| 6 | Dropped null-price rows | ~247 rows removed |
| 7 | Filled numeric nulls | Median for skewed cols, 0 for review counts |
| 8 | Filled text nulls | `name` → "Unnamed Listing", `host_name` → "Unknown", etc. |
| 9 | Filled categorical nulls | `instant_bookable` → mode, `cancellation_policy` → mode |
| 10 | Filled `country`/`country_code` | "United States" / "US" (all NYC data) |
| 11 | Dropped unfillable rows | ~50 rows missing lat/long/neighbourhood |
| 12 | Standardized column names | lowercase + underscores |
| 13 | Removed bogus dates | `1900-01-01` → `NaT` (never-reviewed listings) |
| 14 | Fixed `construction_year` dtype | `float64` → `int64` (no more 2012.0) |
| 15 | Exported | `cleaned_airbnb_final.csv` — **Tableau-ready** 🎯 |